In [ ]:
# Install required libraries
!pip install apify-client pandas requests -q

# ALL IMPORTS at the top
from apify_client import ApifyClient
import pandas as pd
import requests
import re
import json
from datetime import datetime

print("🚀 Starting Instagram data extraction...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.4/140.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 29.9 MB/s eta 0:00:00
🚀 Starting Instagram data extraction...


In [ ]:
# ===== CONFIGURE THESE =====
APIFY_API_TOKEN = "apify_api_N2mhX2MTy6bVudG5eo6EFTIvE5G6b72wekLJ"  # Keep your token here
INSTAGRAM_URL = "https://www.instagram.com/mco_wonders.pk/"  # Use the full URL

# Initialize the client
client = ApifyClient(APIFY_API_TOKEN)

# ✅ CORRECTED INPUT FORMAT - Using directUrls instead of usernames
run_input = {
    "directUrls": [INSTAGRAM_URL],  # This is the key change
    "resultsType": "posts",
    "resultsLimit": 50,
}

# ✅ CORRECT ACTOR NAME - This is the working Instagram scraper
print(f"🚀 Scraping {INSTAGRAM_URL}...")
run = client.actor("shu8hvrXbJbY3Eb9W").call(run_input=run_input)  # Different Actor ID

print("✅ Scraper started successfully! Fetching data...")

# Get the results
results = []
for item in client.dataset(run["defaultDatasetId"]).iterate_items():
    results.append(item)

print(f"✅ Successfully scraped {len(results)} items!")

🚀 Scraping https://www.instagram.com/mco_wonders.pk/...


[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> Status: RUNNING, Message: 
[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> 2026-05-20T06:21:01.136Z ACTOR: Pulling container image of build MAnd1YgavsPfjAUCE from registry.
[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> 2026-05-20T06:21:01.140Z ACTOR: Creating container.
[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> 2026-05-20T06:21:01.322Z ACTOR: Starting container.
[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> 2026-05-20T06:21:01.324Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> 2026-05-20T06:21:02.213Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> 2026-05-20T06:21:02.354Z INFO  Results Limit 50, ACTOR_MAX_PAID_DATASET_ITEMS
[apify.instagram-scraper runId:XsOh19yTR5bfovBfP] -> 2026-05-20T06:21:02.360Z INFO  [St

✅ Scraper started successfully! Fetching data...
✅ Successfully scraped 50 items!


In [ ]:
# Install required libraries
!pip install apify-client pandas -q

from apify_client import ApifyClient
import pandas as pd
from datetime import datetime

# ============================================
# YOUR APIFY TOKEN
# ============================================
APIFY_TOKEN = "apify_api_N2mhX2MTy6bVudG5eo6EFTIvE5G6b72wekLJ"
client = ApifyClient(APIFY_TOKEN)

print("🚀 Starting Instagram data extraction...")
print("="*50)

# ============================================
# PART 1: Get PROFILE DATA (using working actor)
# ============================================
print("\n📊 Fetching PROFILE data...")

profile_run = client.actor("apify/instagram-profile-scraper").call(
    run_input={"usernames": ["mco_wonders.pk"]}
)

profile_items = list(client.dataset(profile_run["defaultDatasetId"]).iterate_items())

if not profile_items:
    print("❌ Failed to get profile data")
    exit()

p = profile_items[0]

# Extract profile metrics
profile_data = {
    'username': p.get('username', p.get('userName', 'mco_wonders.pk')),
    'full_name': p.get('fullName', p.get('full_name', '')),
    'bio': p.get('biography', p.get('bio', '')),
    'followers': p.get('followersCount', p.get('followers', 0)),
    'following': p.get('followsCount', p.get('following', 0)),
    'posts_count': p.get('postsCount', p.get('mediaCount', 0)),
    'is_verified': p.get('verified', p.get('isVerified', False)),
    'is_private': p.get('isPrivate', False),
}

print(f"   ✅ @{profile_data['username']} - {profile_data['followers']:,} followers")

# ============================================
# PART 2: Get POSTS DATA (engagement metrics)
# ============================================
print("\n📸 Fetching POSTS data...")

posts_run = client.actor("shu8hvrXbJbY3Eb9W").call(
    run_input={
        "directUrls": ["https://www.instagram.com/mco_wonders.pk/"],
        "resultsType": "posts",
        "resultsLimit": 50,
    }
)

posts_items = list(client.dataset(posts_run["defaultDatasetId"]).iterate_items())

# Extract posts
all_posts = []
post_timestamps = []

for item in posts_items:
    if 'likesCount' in item:  # This is a post
        # Parse timestamp
        timestamp_str = item.get('timestamp', '')
        post_date = None
        if timestamp_str:
            try:
                if 'T' in timestamp_str:
                    post_date = datetime.fromisoformat(timestamp_str.replace('Z', '+00:00'))
                else:
                    post_date = datetime.fromtimestamp(int(timestamp_str))
                post_timestamps.append(post_date)
            except:
                pass

        post = {
            'post_url': item.get('url', ''),
            'likes': item.get('likesCount', 0),
            'comments': item.get('commentsCount', 0),
            'views': item.get('videoViewCount', 0) if item.get('__typename') == 'GraphVideo' else None,
            'timestamp': timestamp_str,
            'post_date': post_date.strftime('%Y-%m-%d') if post_date else 'N/A',
            'caption': item.get('caption', '')[:100],
        }
        all_posts.append(post)

print(f"   ✅ Retrieved {len(all_posts)} posts")

# ============================================
# PART 3: Calculate all metrics
# ============================================

# Date calculations
if post_timestamps:
    first_post_date = min(post_timestamps).date()
    last_post_date = max(post_timestamps).date()
else:
    first_post_date = last_post_date = "Not available"

# Engagement totals
total_likes = sum(post['likes'] for post in all_posts)
total_comments = sum(post['comments'] for post in all_posts)
total_views = sum(post['views'] for post in all_posts if post['views'])

# Engagement rate
if profile_data['followers'] > 0 and all_posts:
    avg_interactions = (total_likes + total_comments) / len(all_posts)
    engagement_rate = (avg_interactions / profile_data['followers']) * 100
else:
    engagement_rate = 0

# Best post
if all_posts:
    best_post = max(all_posts, key=lambda x: x['likes'] + x['comments'])
else:
    best_post = None

# ============================================
# PART 4: DISPLAY FINAL REPORT
# ============================================
print("\n" + "="*60)
print("📸 INSTAGRAM ACCOUNT ANALYTICS REPORT")
print("="*60)

print("\n🏷️  ACCOUNT DETAILS:")
print(f"   Username: @{profile_data['username']}")
print(f"   Name: {profile_data['full_name'] or 'Not available'}")
print(f"   Bio: {profile_data['bio'][:100] if profile_data['bio'] else 'Not available'}")
print(f"   Private Account: {'Yes' if profile_data['is_private'] else 'No'}")
print(f"   Verified: {'Yes' if profile_data['is_verified'] else 'No'}")

print("\n📊 PROFILE METRICS:")
print(f"   👥 Total Followers: {profile_data['followers']:,}")
print(f"   ➕ Total Following: {profile_data['following']:,}")
print(f"   📸 Total Posts: {profile_data['posts_count']}")

print("\n📅 POST TIMELINE:")
print(f"   🆕 First Post: {first_post_date}")
print(f"   🔄 Last Post: {last_post_date}")
print(f"   📝 Posts Analyzed: {len(all_posts)}")

print("\n❤️ ENGAGEMENT TOTALS:")
print(f"   Total Likes: {total_likes:,}")
print(f"   Total Comments: {total_comments:,}")
if total_views > 0:
    print(f"   Total Views: {total_views:,}")

print("\n📈 ENGAGEMENT RATE:")
print(f"   Average interactions per post: {avg_interactions:.1f}" if all_posts else "   N/A")
print(f"   Engagement Rate: {engagement_rate:.2f}% (interactions/follower)")

if best_post:
    print("\n🏆 BEST PERFORMING POST:")
    print(f"   🔗 {best_post['post_url']}")
    print(f"   ❤️ {best_post['likes']:,} likes | 💬 {best_post['comments']:,} comments")
    if best_post.get('views'):
        print(f"   👁️ {best_post['views']:,} views")
    print(f"   📅 Date: {best_post['post_date']}")
    if best_post['caption']:
        print(f"   📝 Caption: {best_post['caption'][:80]}...")

print("\n" + "="*60)

# ============================================
# PART 5: SAVE TO CSV FILES
# ============================================

# Save profile data
df_profile = pd.DataFrame([profile_data])
df_profile.to_csv('mco_wonders_profile.csv', index=False)
print(f"\n💾 Profile data saved to 'mco_wonders_profile.csv'")

# Save posts data
df_posts = pd.DataFrame(all_posts)
df_posts.to_csv('mco_wonders_posts.csv', index=False)
print(f"💾 Posts data saved to 'mco_wonders_posts.csv'")

# Create summary report for instructor
summary_data = {
    'Metric': [
        'Username', 'Full Name', 'Followers', 'Following', 'Total Posts',
        'First Post Date', 'Last Post Date', 'Posts Analyzed',
        'Total Likes', 'Total Comments', 'Engagement Rate (%)'
    ],
    'Value': [
        f"@{profile_data['username']}",
        profile_data['full_name'],
        f"{profile_data['followers']:,}",
        f"{profile_data['following']:,}",
        profile_data['posts_count'],
        str(first_post_date),
        str(last_post_date),
        len(all_posts),
        f"{total_likes:,}",
        f"{total_comments:,}",
        f"{engagement_rate:.2f}"
    ]
}
df_summary = pd.DataFrame(summary_data)
df_summary.to_csv('instagram_analysis_summary.csv', index=False)
print(f"💾 Summary report saved to 'instagram_analysis_summary.csv'")

print("\n✅ All data extracted successfully!")

🚀 Starting Instagram data extraction...

📊 Fetching PROFILE data...


[apify.instagram-profile-scraper runId:zfFtEa9rt1h0Xiafo] -> Status: RUNNING, Message: 
[apify.instagram-profile-scraper runId:zfFtEa9rt1h0Xiafo] -> 2026-05-20T03:16:06.235Z ACTOR: Pulling container image of build feJJhOsEkgEmfb7hG from registry.
[apify.instagram-profile-scraper runId:zfFtEa9rt1h0Xiafo] -> 2026-05-20T03:16:06.237Z ACTOR: Creating container.
[apify.instagram-profile-scraper runId:zfFtEa9rt1h0Xiafo] -> 2026-05-20T03:16:06.767Z ACTOR: Starting container.
[apify.instagram-profile-scraper runId:zfFtEa9rt1h0Xiafo] -> 2026-05-20T03:16:06.768Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-profile-scraper runId:zfFtEa9rt1h0Xiafo] -> 2026-05-20T03:16:07.393Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.instagram-profile-scraper runId:zfFtEa9rt1h0Xiafo] -> 2026-05-20T03:16:07.515Z INFO  Results Limit undefined, ACTOR_MAX_PAID_DATASET_ITEMS
[apify.instagram-profile

   ✅ @mco_wonders.pk - 621 followers

📸 Fetching POSTS data...


[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> Status: RUNNING, Message: 
[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> 2026-05-20T03:16:18.551Z ACTOR: Pulling container image of build MAnd1YgavsPfjAUCE from registry.
[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> 2026-05-20T03:16:18.553Z ACTOR: Creating container.
[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> 2026-05-20T03:16:18.595Z ACTOR: Starting container.
[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> 2026-05-20T03:16:18.597Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> 2026-05-20T03:16:19.200Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> 2026-05-20T03:16:19.311Z INFO  Results Limit 50, ACTOR_MAX_PAID_DATASET_ITEMS
[apify.instagram-scraper runId:TMdqsKnJMauOKHPXE] -> 2026-05-20T03:16:19.314Z INFO  [St

   ✅ Retrieved 50 posts

📸 INSTAGRAM ACCOUNT ANALYTICS REPORT

🏷️  ACCOUNT DETAILS:
   Username: @mco_wonders.pk
   Name: MCO_wonders.pk
   Bio: I'm a small business owner🤍. 
A girl with multiple talents🌸. 
Your one follow can give me a hope to 
   Private Account: No
   Verified: No

📊 PROFILE METRICS:
   👥 Total Followers: 621
   ➕ Total Following: 1
   📸 Total Posts: 53

📅 POST TIMELINE:
   🆕 First Post: 2025-08-17
   🔄 Last Post: 2026-04-09
   📝 Posts Analyzed: 50

❤️ ENGAGEMENT TOTALS:
   Total Likes: 3,209
   Total Comments: 576

📈 ENGAGEMENT RATE:
   Average interactions per post: 75.7
   Engagement Rate: 12.19% (interactions/follower)

🏆 BEST PERFORMING POST:
   🔗 https://www.instagram.com/p/DSE62hlDO6p/
   ❤️ 1,875 likes | 💬 128 comments
   📅 Date: 2025-12-10
   📝 Caption: #explorepage  #explorereels #trendingreels...


💾 Profile data saved to 'mco_wonders_profile.csv'
💾 Posts data saved to 'mco_wonders_posts.csv'
💾 Summary report saved to 'instagram_analysis_summary.csv'

✅ A

In [ ]:
from apify_client import ApifyClient
import pandas as pd
from google.colab import files

APIFY_TOKEN = "apify_api_N2mhX2MTy6bVudG5eo6EFTIvE5G6b72wekLJ"  # ← paste your token
REEL_URL    = "https://www.instagram.com/reel/DRXBxeXjDS8/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="

client = ApifyClient(APIFY_TOKEN)

print("⏳ Fetching reel data... (~45 seconds)")

run = client.actor("apify/instagram-scraper").call(
    run_input={
        "directUrls"    : [REEL_URL],
        "resultsType"   : "posts",
        "resultsLimit"  : 1,
        "addParentData" : False
    }
)

items = list(client.dataset(run["defaultDatasetId"]).iterate_items())

if items:
    r = items[0]

    print("\n🔑 Keys returned:", list(r.keys()))  # helps debug

    views    = r.get("videoPlayCount", r.get('videoPlayCount',  0))
    likes    = r.get("likesCount",     r.get("likes",          0))
    comments = r.get("commentsCount",  r.get("comments",       0))
    shares   = r.get("sharesCount",    r.get("shares",          "Not public"))
    saves    = r.get("savesCount",     r.get("saves",           "Not public"))
    reposts  = r.get("repostsCount",   r.get("reshares",        "Not public"))
    posted   = r.get("timestamp",      r.get("taken_at",        "N/A"))
    caption  = r.get("caption",        r.get("text",            ""))

    print("\n✅ REEL METRICS")
    print("="*45)
    print(f"  🔗 URL      : {REEL_URL}")
    print(f"  📅 Posted   : {posted}")
    print(f"  👁️  Views    : {views:,}"    if isinstance(views, int)    else f"  👁️  Views    : {views}")
    print(f"  ❤️  Likes    : {likes:,}"    if isinstance(likes, int)    else f"  ❤️  Likes    : {likes}")
    print(f"  💬 Comments : {comments:,}" if isinstance(comments, int) else f"  💬 Comments : {comments}")
    print(f"  🔁 Shares   : {shares}")
    print(f"  🔖 Saves    : {saves}")
    print(f"  ♻️  Reposts  : {reposts}")
    print(f"  📝 Caption  : {str(caption)[:100]}")
    print("="*45)

    # Save to CSV
    df_reel = pd.DataFrame([
        {"Metric": "Reel URL",  "Value": REEL_URL},
        {"Metric": "Posted On", "Value": posted},
        {"Metric": "Views",     "Value": views},
        {"Metric": "Likes",     "Value": likes},
        {"Metric": "Comments",  "Value": comments},
        {"Metric": "Shares",    "Value": shares},
        {"Metric": "Saves",     "Value": saves},
        {"Metric": "Reposts",   "Value": reposts},
        {"Metric": "Caption",   "Value": caption},
    ])

    df_reel.to_csv("reel_metrics.csv", index=False)
    files.download("reel_metrics.csv")
    print("\n💾 reel_metrics.csv downloaded!")

else:
    print("❌ Empty result. Try the backup actor below.")

⏳ Fetching reel data... (~45 seconds)


[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> Status: RUNNING, Message: 
[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> 2026-05-20T05:57:49.069Z ACTOR: Pulling container image of build MAnd1YgavsPfjAUCE from registry.
[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> 2026-05-20T05:57:49.071Z ACTOR: Creating container.
[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> 2026-05-20T05:57:49.178Z ACTOR: Starting container.
[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> 2026-05-20T05:57:49.180Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> 2026-05-20T05:57:49.824Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> 2026-05-20T05:57:49.947Z INFO  Results Limit 1, ACTOR_MAX_PAID_DATASET_ITEMS
[apify.instagram-scraper runId:f9CQ2D5MQgW2d8o6Y] -> 2026-05-20T05:57:49.952Z INFO  [Sta


🔑 Keys returned: ['inputUrl', 'id', 'type', 'shortCode', 'caption', 'hashtags', 'mentions', 'url', 'commentsCount', 'firstComment', 'latestComments', 'dimensionsHeight', 'dimensionsWidth', 'displayUrl', 'images', 'videoUrl', 'audioUrl', 'alt', 'likesCount', 'videoViewCount', 'videoPlayCount', 'timestamp', 'childPosts', 'ownerFullName', 'ownerUsername', 'ownerId', 'productType', 'videoDuration', 'isPinned', 'musicInfo', 'isCommentsDisabled']

✅ REEL METRICS
  🔗 URL      : https://www.instagram.com/reel/DRXBxeXjDS8/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
  📅 Posted   : 2025-11-22T13:07:39.000Z
  👁️  Views    : 996
  ❤️  Likes    : 25
  💬 Comments : 7
  🔁 Shares   : Not public
  🔖 Saves    : Not public
  ♻️  Reposts  : Not public
  📝 Caption  : Jaldi sy comment karo🥰🤍. 
#explore #explorereels
#trendingreels


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


💾 reel_metrics.csv downloaded!


In [ ]:
from apify_client import ApifyClient
import pandas as pd
from google.colab import files

APIFY_TOKEN = "apify_api_N2mhX2MTy6bVudG5eo6EFTIvE5G6b72wekLJ"
REEL_URL = "https://www.instagram.com/reel/DSE62hlDO6p/"

client = ApifyClient(APIFY_TOKEN)

print("⏳ Fetching comments... (~30 seconds)")

# ✅ FIXED: Use 'directUrls' instead of 'postUrls'
run = client.actor("apify/instagram-comment-scraper").call(
    run_input={
        "directUrls": [REEL_URL],    # Changed from 'postUrls' to 'directUrls'
        "resultsLimit": 128
    }
)

comments = list(client.dataset(run["defaultDatasetId"]).iterate_items())

print(f"\n✅ Total comments fetched: {len(comments)}")
print("="*55)

for i, c in enumerate(comments[:10], 1):
    user = c.get("ownerUsername", c.get("username", "unknown"))
    text = c.get("text", c.get("comment", ""))
    likes = c.get("likesCount", c.get("likes", 0))
    timestamp = c.get("timestamp", "N/A")
    print(f"{i:>3}. @{user}")
    print(f"      💬 {text[:80]}")
    print(f"      ❤️  {likes} likes  |  🕐 {timestamp}")
    print()

if len(comments) > 10:
    print(f"  ... and {len(comments) - 10} more comments in the CSV")

print("="*55)

# Save all comments to CSV
df_comments = pd.DataFrame([{
    "username": c.get("ownerUsername", c.get("username", "")),
    "comment": c.get("text", c.get("comment", "")),
    "likes": c.get("likesCount", c.get("likes", 0)),
    "timestamp": c.get("timestamp", "N/A"),
    "replies": c.get("repliesCount", 0),
} for c in comments])

df_comments.to_csv("reel_comments.csv", index=False)
files.download("reel_comments.csv")
print("\n💾 All comments saved to reel_comments.csv")

⏳ Fetching comments... (~30 seconds)


[apify.instagram-comment-scraper runId:yMdQVb1KoMCems3eE] -> Status: RUNNING, Message: 
[apify.instagram-comment-scraper runId:yMdQVb1KoMCems3eE] -> 2026-05-20T07:53:32.049Z ACTOR: Pulling container image of build GvC9HniHmPGyGtA1G from registry.
[apify.instagram-comment-scraper runId:yMdQVb1KoMCems3eE] -> 2026-05-20T07:53:32.052Z ACTOR: Creating container.
[apify.instagram-comment-scraper runId:yMdQVb1KoMCems3eE] -> 2026-05-20T07:53:32.094Z ACTOR: Starting container.
[apify.instagram-comment-scraper runId:yMdQVb1KoMCems3eE] -> 2026-05-20T07:53:32.095Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-comment-scraper runId:yMdQVb1KoMCems3eE] -> 2026-05-20T07:53:32.643Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.instagram-comment-scraper runId:yMdQVb1KoMCems3eE] -> 2026-05-20T07:53:32.757Z INFO  Results Limit 128, ACTOR_MAX_PAID_DATASET_ITEMS
[apify.instagram-comment-scrap


✅ Total comments fetched: 15
  1. @mco_wonders.pk
      💬 Don't forget to support my small business by follow me on tiktok and on Instagra
      ❤️  23 likes  |  🕐 2025-12-28T06:51:56.000Z

  2. @zeeshan54658
      💬 Mujhy bhi bnwany hain
      ❤️  4 likes  |  🕐 2025-12-27T22:00:58.000Z

  3. @hafsa018368
      💬 Tell me plz
      ❤️  2 likes  |  🕐 2026-04-11T03:10:22.000Z

  4. @itzz.khadija_11
      💬 From where 😢
      ❤️  3 likes  |  🕐 2025-12-12T06:20:02.000Z

  5. @botanique_beautyksa
      💬 Where are they located?
      ❤️  1 likes  |  🕐 2026-04-27T22:34:57.000Z

  6. @ashhhyy__30
      💬 Besttttt of luckkkkkk..!! ❤️
      ❤️  10 likes  |  🕐 2025-12-10T08:58:28.000Z

  7. @mus_kanshahzadi
      💬 ماشاءاللہ ماشاءاللہ❤️🔥 zabardast
      ❤️  5 likes  |  🕐 2025-12-10T13:12:48.000Z

  8. @abeer_khan_8899
      💬 Id please
      ❤️  2 likes  |  🕐 2025-12-27T16:52:36.000Z

  9. @hafsa.abaid
      💬 What’s the size ?
      ❤️  1 likes  |  🕐 2025-12-23T18:23:26.000Z

 10. @mallqai
    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


💾 All comments saved to reel_comments.csv


In [ ]:
# 1. INSTALL LIBRARIES
!pip install apify-client pandas -q

from apify_client import ApifyClient
import pandas as pd
import time
from google.colab import files

# 2. CONFIGURATION
APIFY_TOKEN = "apify_api_N2mhX2MTy6bVudG5eo6EFTIvE5G6b72wekLJ"
USERNAME = "mco_wonders.pk"
client = ApifyClient(APIFY_TOKEN)

print(f"🚀 Starting Auto-Extraction for @{USERNAME}...")

# 3. FETCH DATA
# We will pull from the main profile but look for video content (Reels)
reels_run = client.actor("apify/instagram-scraper").call(
    run_input={
        "directUrls": [f"https://www.instagram.com/{USERNAME}/"],
        "resultsLimit": 30, # Fetch more so we can find at least 20 reels
        "resultsType": "posts"
    }
)

dataset_id = reels_run.default_dataset_id
items = list(client.dataset(dataset_id).iterate_items())

# Filter for anything that is a Video or specifically a Reel
reels_list = [
    item for item in items
    if item.get('type') == 'Video' or item.get('productType') == 'clips' or '/reel/' in item.get('url', '')
]

# --- FALLBACK MECHANISM ---
# If Instagram blocks the scroll and returns 0, we use your manual links from Part 2
# to ensure you still get a consolidated CSV for the exam.
if len(reels_list) == 0:
    print("⚠️ Auto-scroll blocked by Instagram. Using your manually identified links to proceed...")
    manual_links = [
        "https://www.instagram.com/reel/DNPwi5VMj7-/",
        "https://www.instagram.com/reel/DNm6-gVMa1y/",
        "https://www.instagram.com/reel/DOEXMVxDBG1/",
        "https://www.instagram.com/reel/DOG4PpiDPWm/",
        "https://www.instagram.com/reel/DOd5v0TjKui/",
        "https://www.instagram.com/reel/DPEfje8jCb2/",
        "https://www.instagram.com/reel/DP9KVEjjMkS/",
        "https://www.instagram.com/reel/DQte7WkDGhO/",
        "https://www.instagram.com/reel/DRCkbL6jJFY/",
        "https://www.instagram.com/reel/DRoqgPeDLM0/",
        "https://www.instagram.com/reel/DSE62hlDO6p/",
        "https://www.instagram.com/reel/DSc066Ak4G9/",
        "http://instagram.com/reel/DTLKH0EjAiU/",
        "https://www.instagram.com/reel/DTsSMG9jPx9/",
        "https://www.instagram.com/reel/DUf_j3ujHXS/",
        "https://www.instagram.com/reel/DU6EArwDInc/",
        "https://www.instagram.com/reel/DVOlJOxDM7s/",
        "https://www.instagram.com/reel/DV35K95jOe3/",
        "https://www.instagram.com/reel/DWWzG-vDAI_/",
        "https://www.instagram.com/reel/DW6WNFWjBvc/"
    ]
    # Create dummy reel objects to feed into the comment scraper
    reels_list = [{"url": link} for link in manual_links]

print(f"✅ Ready to process {len(reels_list)} reels.")

# 4. LOOP AND EXTRACT COMMENTS
master_records = []

for i, reel in enumerate(reels_list[:20]): # Limit to top 20
    reel_url = reel.get('url')
    print(f"🔗 [{i+1}/20] Extracting Comments from: {reel_url}")

    try:
        comment_run = client.actor("apify/instagram-comment-scraper").call(
            run_input={
                "directUrls": [reel_url],
                "resultsLimit": 15
            }
        )

        c_dataset_id = comment_run.default_dataset_id
        comments = list(client.dataset(c_dataset_id).iterate_items())

        if not comments:
            master_records.append({
                "Reel_URL": reel_url,
                "Views": reel.get('videoViewCount', "N/A"),
                "Likes": reel.get('likesCount', "N/A"),
                "Comment_Text": "No comments found",
                "Comment_User": "N/A"
            })
        else:
            for c in comments:
                master_records.append({
                    "Reel_URL": reel_url,
                    "Views": reel.get('videoViewCount', "N/A"),
                    "Likes": reel.get('likesCount', "N/A"),
                    "Comment_Text": c.get('text', ''),
                    "Comment_User": c.get('ownerUsername', 'unknown')
                })

    except Exception as e:
        print(f"⚠️ Error on reel {i+1}: {e}")
        continue

    time.sleep(1)

# 5. SAVE AND DOWNLOAD
if master_records:
    df = pd.DataFrame(master_records)
    filename = f"{USERNAME}_consolidated_reels.csv"
    df.to_csv(filename, index=False)
    print(f"\n✅ SUCCESS! {len(df)} rows collected.")
    files.download(filename)
else:
    print("❌ Critical Error: Could not extract any data.")

🚀 Starting Auto-Extraction for @mco_wonders.pk...


[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> Status: RUNNING, Message: 
[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> 2026-06-25T18:00:21.420Z ACTOR: Pulling container image of build EPybSjX6ikJZeCsE1 from registry.
[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> 2026-06-25T18:00:21.422Z ACTOR: Creating container.
[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> 2026-06-25T18:00:21.746Z ACTOR: Starting container.
[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> 2026-06-25T18:00:21.748Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> 2026-06-25T18:00:22.463Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.18.0"}
[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> 2026-06-25T18:00:23.646Z INFO  [Status message]: Starting the scraper with 1 direct URL(s)
[apify.instagram-scraper runId:sccGFzvP0Fr5QtIUi] -> Status: RUNNING, Mess

✅ Ready to process 26 reels.
🔗 [1/20] Extracting Comments from: https://www.instagram.com/p/DW6WNFWjBvc/


[apify.instagram-comment-scraper runId:J62mHSXa5qU0hm8bm] -> Status: RUNNING, Message: 
[apify.instagram-comment-scraper runId:J62mHSXa5qU0hm8bm] -> 2026-06-25T18:02:00.129Z ACTOR: Pulling container image of build XhAwHK5aZQYG1YHjO from registry.
[apify.instagram-comment-scraper runId:J62mHSXa5qU0hm8bm] -> 2026-06-25T18:02:00.135Z ACTOR: Creating container.
[apify.instagram-comment-scraper runId:J62mHSXa5qU0hm8bm] -> 2026-06-25T18:02:00.194Z ACTOR: Starting container.
[apify.instagram-comment-scraper runId:J62mHSXa5qU0hm8bm] -> 2026-06-25T18:02:00.196Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-comment-scraper runId:J62mHSXa5qU0hm8bm] -> 2026-06-25T18:02:00.851Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.18.0"}
[apify.instagram-comment-scraper runId:J62mHSXa5qU0hm8bm] -> 2026-06-25T18:02:00.990Z INFO  [Status message]: Starting the scraper with 1 direct URL(s)
[apify.instagram-c

🔗 [2/20] Extracting Comments from: https://www.instagram.com/p/DU6EArwDInc/


...

🔗 [20/20] Extracting Comments from: https://www.instagram.com/p/DV35K95jOe3/


[apify.instagram-comment-scraper runId:82f4pkxp2zQOgWILm] -> Status: RUNNING, Message: 
[apify.instagram-comment-scraper runId:82f4pkxp2zQOgWILm] -> 2026-06-25T18:06:56.812Z ACTOR: Pulling container image of build XhAwHK5aZQYG1YHjO from registry.
[apify.instagram-comment-scraper runId:82f4pkxp2zQOgWILm] -> 2026-06-25T18:06:56.814Z ACTOR: Creating container.
[apify.instagram-comment-scraper runId:82f4pkxp2zQOgWILm] -> 2026-06-25T18:06:56.880Z ACTOR: Starting container.
[apify.instagram-comment-scraper runId:82f4pkxp2zQOgWILm] -> 2026-06-25T18:06:56.882Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-comment-scraper runId:82f4pkxp2zQOgWILm] -> 2026-06-25T18:06:57.445Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.18.0"}
[apify.instagram-comment-scraper runId:82f4pkxp2zQOgWILm] -> 2026-06-25T18:06:57.560Z INFO  [Status message]: Starting the scraper with 1 direct URL(s)
[apify.instagram-c


✅ SUCCESS! 151 rows collected.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import whisper
import moviepy.editor as mp
import pandas as pd
from google.colab import files

# 1. LOAD WHISPER (Base model is good for Roman Urdu/English mix)
print("⏳ Loading AI Model... please wait.")
model = whisper.load_model("base")

def transcribe_all_reels():
    data = []
    # Search specifically in the Google Colab content folder
    path = '/content/'
    all_files = os.listdir(path)

    # This finds all .mp4 files regardless of their names (handles spaces automatically)
    video_files = [f for f in all_files if f.endswith('.mp4')]

    if not video_files:
        print("❌ No .mp4 files found! Make sure you see them in the folder icon on the left.")
        return None

    print(f"✅ Found {len(video_files)} reels: {video_files}")

    for video_file in video_files:
        try:
            # Use the FULL PATH (/content/filename.mp4)
            full_video_path = os.path.join(path, video_file)
            print(f"\n⌛ Currently Processing: {video_file}")

            # Extract Audio using MoviePy
            clip = mp.VideoFileClip(full_video_path)
            audio_path = "temp_audio.mp3"
            clip.audio.write_audiofile(audio_path, logger=None)

            # Transcribe with Whisper
            print(f"   📝 Generating Transcript...")
            result = model.transcribe(audio_path)
            transcript = result['text']

            # Store results
            data.append({
                "Filename": video_file,
                "Transcript": transcript,
                "Status": "Success"
            })

            # Clean up files and memory
            clip.close()
            os.remove(audio_path)

        except Exception as e:
            print(f"⚠️ Error on {video_file}: {e}")
            data.append({"Filename": video_file, "Transcript": "ERROR", "Status": str(e)})
            continue

    return pd.DataFrame(data)

# --- EXECUTE ---
df_results = transcribe_all_reels()

if df_results is not None:
    # Save the result to CSV
    output_name = "reels_transcripts_final.csv"
    df_results.to_csv(output_name, index=False)
    print(f"\n🎉 ALL DONE! Your transcripts are saved in {output_name}")

    # Show a preview of the data
    print("\n--- DATA PREVIEW ---")
    print(df_results[['Filename', 'Transcript']].head())

    # Download the file automatically
    files.download(output_name)

⏳ Loading AI Model... please wait.
✅ Found 6 reels: ['Chocolate boquet🍫💐. DM to place your order..!!! 🥰🤍. . . . . . . . . . . . . . . . . . . #exp.mp4', 'Why 😭😭🥺🥺. . . . . . . . . #viralreels #instamood #love #explore #foryou #support #instagram .mp4', 'ITS OUR SECOND COLLECTION 😍🤍. Instagram bhai-behan kar dey viral meri bhi video😩😭. #viralre.mp4', 'Just few days are left😍🤍🎀. #comingsoon #viralreels #smallbusiness #smallbusinessowners #small_2.mp4', 'DM TO PACE YOUR ORDER... !! #growoninstagram #dailygrowth #fypage #explore #explorepage #instamo.mp4', 'Just few days are left😍🤍🎀. #comingsoon #viralreels #smallbusiness #smallbusinessowners #small.mp4']

⌛ Currently Processing: Chocolate boquet🍫💐. DM to place your order..!!! 🥰🤍. . . . . . . . . . . . . . . . . . . #exp.mp4


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



   📝 Generating Transcript...

⌛ Currently Processing: Why 😭😭🥺🥺. . . . . . . . . #viralreels #instamood #love #explore #foryou #support #instagram .mp4


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



   📝 Generating Transcript...

⌛ Currently Processing: ITS OUR SECOND COLLECTION 😍🤍. Instagram bhai-behan kar dey viral meri bhi video😩😭. #viralre.mp4


...

   📝 Generating Transcript...

⌛ Currently Processing: Just few days are left😍🤍🎀. #comingsoon #viralreels #smallbusiness #smallbusinessowners #small.mp4


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



   📝 Generating Transcript...

🎉 ALL DONE! Your transcripts are saved in reels_transcripts_final.csv

--- DATA PREVIEW ---
                                            Filename  \
0  Chocolate boquet🍫💐. DM to place your order..!!...   
1  Why 😭😭🥺🥺. . . . . . . . . #viralreels #instamo...   
2  ITS OUR SECOND COLLECTION 😍🤍. Instagram bhai-b...   
3  Just few days are left😍🤍🎀. #comingsoon #viralr...   
4  DM TO PACE YOUR ORDER... !! #growoninstagram #...   

                                          Transcript  
0   Agar apne je buke nahi deka taa apne zindagi ...  
1   CLEAN-AINNão  subsidies near towards the tick...  
2   Our second collective. I feel like I'm in a m...  
3   اور پھر اچانک سے ایک دن تمہاری باری آئی اور ت...  
4   Our first collection! I thought that was a go...  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>